# 102 - Proyecto — **Soluciones + Autotests**

In [ ]:

import json
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

# Datos sintéticos
rng = np.random.default_rng(0)
df = pd.DataFrame({
    "feature1": rng.normal(0,1,300),
    "feature2": rng.uniform(-1,1,300),
    "target": (rng.random(300) > 0.5).astype(int)
})

X = df[["feature1","feature2"]]
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

clf = LogisticRegression(max_iter=200).fit(X_train_s, y_train)
pred = clf.predict(X_test_s)
acc = accuracy_score(y_test, pred)

Path("artifacts").mkdir(exist_ok=True)
joblib.dump({"scaler": scaler, "model": clf}, "artifacts/model.joblib")
with open("artifacts/metrics.json","w") as f:
    json.dump({"accuracy": float(acc)}, f)


In [ ]:

# Helper de autotests
def check(cond, ok_msg, fail_msg):
    if cond:
        print("✔", ok_msg)
    else:
        raise AssertionError("✘ " + fail_msg)


In [ ]:

# Tests
from pathlib import Path, PurePath
check(0.3 <= acc <= 0.8, "Accuracy dentro de un rango razonable (sintético).", "Accuracy fuera de rango esperado.")
check(Path("artifacts/model.joblib").exists(), "Modelo exportado en artifacts.", "No se encontró artifacts/model.joblib.")
check(Path("artifacts/metrics.json").exists(), "Métricas guardadas en artifacts.", "No se encontró artifacts/metrics.json.")
